In [ ]:
import os
import json
import time
import random
from typing import List, Dict, Tuple
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import openai

# Task 5 cell 0: Auto Tagging Support Tickets Using LLM
# GitHub Copilot


# Requires: set OPENAI_API_KEY env var

openai.api_key = os.getenv("OPENAI_API_KEY")

# ---- Data loading helpers ----
def load_tickets(path: str = "tickets.csv", text_col: str = "text", tag_col: str = "tags") -> pd.DataFrame:
    df = pd.read_csv(path)
    df = df[[text_col, tag_col]].dropna()
    return df.rename(columns={text_col: "text", tag_col: "tags"})

def extract_tag_set(df: pd.DataFrame, tag_col: str = "tags", sep: str = ",") -> List[str]:
    tags = set()
    for t in df[tag_col].astype(str):
        for tag in [x.strip() for x in t.split(sep) if x.strip()]:
            tags.add(tag)
    return sorted(tags)

# ---- Prompt templates ----
def build_zero_shot_prompt(text: str, tag_list: List[str], instruct: str = "") -> str:
    tags_str = ", ".join(tag_list)
    prompt = (
        f"You are a support ticket classifier. Given the ticket text, return the top 3 most probable tags "
        f"from this list: [{tags_str}]. Respond with JSON exactly in this format:\n"
        f'{{"predictions":[{{"tag":"TagA","score":0.81}},{{"tag":"TagB","score":0.12}},{{"tag":"TagC","score":0.07}}]}}\n\n'
        f"Ticket:\n\"\"\"\n{text}\n\"\"\"\n"
    )
    if instruct:
        prompt = instruct.strip() + "\n\n" + prompt
    return prompt

def build_few_shot_prompt(text: str, examples: List[Tuple[str, List[str]]], tag_list: List[str]) -> str:
    tags_str = ", ".join(tag_list)
    ex_str = ""
    for ex_text, ex_tags in examples:
        ex_json = json.dumps([{"tag":t, "score":round(1/len(ex_tags), 2)} for t in ex_tags][:3])
        ex_str += f"Example Ticket:\n\"\"\"\n{ex_text}\n\"\"\"\nExample Prediction JSON:\n{{\"predictions\":{ex_json}}}\n\n"
    prompt = (
        f"You are a support ticket classifier. Use the examples to predict the top 3 most probable tags "
        f"from this list: [{tags_str}]. Respond with JSON exactly as in examples: "
        f'{{"predictions":[{{"tag":"TagA","score":0.81}},{{"tag":"TagB","score":0.12}},{{"tag":"TagC","score":0.07}}]}}\n\n'
        f"{ex_str}"
        f"Ticket:\n\"\"\"\n{text}\n\"\"\"\n"
    )
    return prompt

# ---- OpenAI call helpers ----
def chat_request(prompt: str, model: str = "gpt-3.5-turbo", temperature: float = 0.0, max_tokens: int = 256) -> str:
    # simple retry/backoff
    for attempt in range(5):
        try:
            resp = openai.ChatCompletion.create(
                model=model,
                messages=[{"role":"user","content":prompt}],
                temperature=temperature,
                max_tokens=max_tokens
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            time.sleep(2 ** attempt)
    raise RuntimeError("OpenAI request failed after retries")

def parse_predictions(resp_text: str) -> List[Tuple[str, float]]:
    try:
        j = json.loads(resp_text)
        preds = j.get("predictions", [])
        out = []
        for p in preds[:3]:
            tag = p.get("tag")
            score = float(p.get("score", 0.0))
            out.append((tag, score))
        return out
    except Exception:
        # best-effort parse fallback: find words from response that match a tag and assume descending order
        tokens = [t.strip(" .,-") for t in resp_text.split()]
        return [(t, 0.0) for t in tokens[:3]]

# ---- Fine-tune helpers (prepare jsonl) ----
def prepare_finetune_jsonl(df: pd.DataFrame, out_path: str = "finetune.jsonl", max_examples: int = None, sep: str = ","):
    lines = []
    it = df.itertuples(index=False)
    for i, row in enumerate(it):
        if max_examples and i >= max_examples:
            break
        text = row.text.replace("\n", " ")
        true_tags = [t.strip() for t in str(row.tags).split(sep) if t.strip()]
        # completion should start with a space per OpenAI fine-tune guidance
        completion = " " + json.dumps({"predictions": [{"tag": t, "score": 1.0/len(true_tags)} for t in true_tags[:3]]})
        obj = {"prompt": text + "\n\n###\n\n", "completion": completion}
        lines.append(json.dumps(obj))
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    return out_path

# ---- Evaluation ----
def evaluate_model(df_eval: pd.DataFrame, tag_list: List[str], mode: str = "zero_shot", few_shot_examples: List[Tuple[str, List[str]]] = None, sample_size: int = 200):
    df = df_eval.sample(n=min(sample_size, len(df_eval)), random_state=42)
    y_true = []
    preds_top1 = []
    preds_top3 = []
    for text, tags in tqdm(df[["text", "tags"]].itertuples(index=False), total=len(df)):
        if mode == "zero_shot":
            prompt = build_zero_shot_prompt(text, tag_list)
        elif mode == "few_shot":
            prompt = build_few_shot_prompt(text, few_shot_examples, tag_list)
        else:
            raise ValueError("mode must be zero_shot or few_shot")
        resp = chat_request(prompt)
        pred = parse_predictions(resp)
        pred_tags = [p[0] for p in pred]
        true_tag = [t.strip() for t in str(tags).split(",") if t.strip()]
        y_true.append(true_tag)
        preds_top1.append(pred_tags[0] if pred_tags else None)
        preds_top3.append(pred_tags)
    # compute top1 and top3 accuracy (if any true tag matches)
    top1_correct = [ (preds_top1[i] in y_true[i]) for i in range(len(y_true)) ]
    top3_correct = [ any(p in y_true[i] for p in preds_top3[i]) for i in range(len(y_true)) ]
    return {"top1_accuracy": sum(top1_correct)/len(top1_correct), "top3_accuracy": sum(top3_correct)/len(top3_correct)}

# ---- Quick run example ----
if __name__ == "__main__":
    # load your dataset (expects 'text' and 'tags' columns)
    df = load_tickets("tickets.csv")
    tags = extract_tag_set(df, tag_col="tags")
    train, test = train_test_split(df, test_size=0.2, random_state=42)
    # pick few-shot examples from train (text, [tags])
    few_shot_examples = [(row.text, [t.strip() for t in str(row.tags).split(",") if t.strip()]) for _, row in train.sample(n=5, random_state=1).iterrows()]
    print("Tag set:", tags)
    print("Evaluating zero-shot (sample)...")
    zs_metrics = evaluate_model(test, tags, mode="zero_shot", sample_size=100)
    print("Zero-shot metrics:", zs_metrics)
    print("Evaluating few-shot (sample)...")
    fs_metrics = evaluate_model(test, tags, mode="few_shot", few_shot_examples=few_shot_examples, sample_size=100)
    print("Few-shot metrics:", fs_metrics)
    # prepare fine-tune jsonl (optional)
    prepare_finetune_jsonl(train, out_path="finetune.jsonl", max_examples=5000)
    print("Prepared finetune.jsonl (use OpenAI fine-tune API to create a model).")